# 🧹 db_products.csv — Data Cleaning Notebook
Cleans the product dataset for use in the AI Vision Clothing Recommendation system.

**Steps covered:**
1. Load & inspect
2. Strip HTML from descriptions
3. Fix non-ASCII / unicode characters in occasions & tags
4. Fill remaining nulls
5. Drop useless MRP column
6. Standardize gender & whitespace
7. Final validation & save

## 1. Load & Inspect

In [14]:
import pandas as pd
import re

df = pd.read_csv('../docs/db_products.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head(3)

Shape: (2031, 21)

Columns: ['id', 'product_id', 'designer', 'seller_id', 'sku', 'name', 'hsn_code', 'material', 'color', 'no_of_components', 'barcode', 'short_description', 'description', 'tags', 'gender', 'occasions', 'mrp', 'categories', 'product_url', 'created_at', 'updated_at']


,id,product_id,designer,seller_id,sku,name,hsn_code,material,color,no_of_components,...,short_description,description,tags,gender,occasions,mrp,categories,product_url,created_at,updated_at
0,1,1,Dahlia,82,da23co005,Kora Mirri Co-ord Set,6204,"100% Handwoven Cotton, 100% Handwoven AND Natu...",KORA RED,2,...,<p>A statement flap trousers paired with a chi...,<p>Turn heads with the Mirri Coord Set&mdash;a...,"Handwoven Ikat, Red, Kora, Coord Set, Polka Do...",Women,"Office parties, Date night, Romantic Dinner, A...",0,"Women, Sets, Co-ord set",https://thechicindian.com/product-details-sku/...,23-11-2025 11:26,23-11-2025 11:26
1,2,10,Dahlia,82,da23co004,Indigo Mirri Co-ord Set,6204,"100% Handwoven Cotton, 100% Handwoven AND Natu...",INDIGO KORA,2,...,<p>A statement flap trousers paired with a chi...,<p>Turn heads with the Mirri Coord Set&mdash;a...,"Handwoven Ikat, Red, Kora, Coord Set, Polka Do...",Women,"Office parties, Date night, Romantic Dinner, A...",0,"Women, Sets, Co-ord set",https://thechicindian.com/product-details-sku/...,23-11-2025 11:26,23-11-2025 11:26
2,3,19,Dahlia,82,da23to004,Red Mirri Shirt,6204,100% Handwoven AND Natural Dyed Malkha,RED,1,...,<p>A chic shirt features a striking open-back ...,"<p>Turn heads with the Mirri Cropped Shirt, a ...","Shirt, Malkha, Red, backless, natural dye, women",Women,"Office parties, Date night, Romantic Dinner, A...",0,"Women, Tops, Shirt",https://thechicindian.com/product-details-sku/...,23-11-2025 11:26,23-11-2025 11:26


In [15]:
print('=== NULL COUNTS ===')
print(df.isnull().sum())
print('\n=== GENDER VALUES ===')
print(df['gender'].value_counts())
print('\n=== MRP VALUES ===')
print(df['mrp'].value_counts())

=== NULL COUNTS ===
id                     0
product_id             0
designer               0
seller_id              0
sku                    0
name                   0
hsn_code               0
material              15
color                  0
no_of_components       0
barcode                0
short_description      2
description            0
tags                 157
gender                41
occasions             75
mrp                    0
categories            41
product_url            0
created_at             0
updated_at             0
dtype: int64

=== GENDER VALUES ===
gender
Women     1615
Men        197
Unisex     178
Name: count, dtype: int64

=== MRP VALUES ===
mrp
0    2031
Name: count, dtype: int64


## 2. Strip HTML from Descriptions
Nearly all rows have raw HTML tags (`<p>`, `&mdash;` etc.) in `description` and `short_description`.

In [16]:
def strip_html(text):
    if pd.isna(text):
        return text
    text = re.sub(r'<[^>]+>', '', str(text))          # remove tags
    text = text.replace('&mdash;', '-')
    text = text.replace('&nbsp;', ' ')
    text = text.replace('&rsquo;', "'")
    text = text.replace('&amp;', '&')
    text = text.replace('&ldquo;', '"')
    text = text.replace('&rdquo;', '"')
    text = re.sub(r'&[a-z]+;', '', text)              # remove any remaining entities
    text = re.sub(r'\s+', ' ', text).strip()          # normalize whitespace
    return text

df['description'] = df['description'].apply(strip_html)
df['short_description'] = df['short_description'].apply(strip_html)

# Verify
print('HTML remaining in description:', df['description'].str.contains('<', na=False).sum())
print('HTML remaining in short_description:', df['short_description'].str.contains('<', na=False).sum())
print('\nSample cleaned description:')
print(df['description'].iloc[0][:300])

HTML remaining in description: 0
HTML remaining in short_description: 0

Sample cleaned description:
Turn heads with the Mirri Coord Set-a striking blend of bold design and effortless elegance. Made from naturally dyed Malkha fabric, the statement flap trousers offer a sleek, structured fit with functional pockets for everyday ease. Paired with a chic open-back cropped shirt in breathable natural c


## 3. Fix Non-ASCII / Unicode Characters
319 rows in `occasions` and some in `tags` have curly quotes, em-dashes etc. that break text matching.

In [17]:
def fix_unicode(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = text.replace('\u2019', "'").replace('\u2018', "'")   # curly apostrophes
    text = text.replace('\u201c', '"').replace('\u201d', '"')   # curly quotes
    text = text.replace('\u2013', '-').replace('\u2014', '-')   # en/em dash
    text = text.replace('\u00e9', 'e').replace('\u00e8', 'e')   # accented e
    return text

df['occasions'] = df['occasions'].apply(fix_unicode)
df['tags'] = df['tags'].apply(fix_unicode)

# Verify
remaining = df['occasions'].dropna().str.contains(r'[^\x00-\x7F]', regex=True).sum()
print('Non-ASCII remaining in occasions:', remaining)

Non-ASCII remaining in occasions: 29


## 4. Fill Remaining Nulls

In [18]:
null_fills = {
    'gender': 'Unknown',
    'occasions': 'Unknown',
    'categories': 'Unknown',
    'tags': 'Unknown',
    'material': 'Unknown',
    'short_description': 'Unknown'
}

for col, val in null_fills.items():
    before = df[col].isna().sum()
    df[col] = df[col].fillna(val)
    print(f'{col}: filled {before} nulls')

print('\nTotal nulls remaining:', df.isnull().sum().sum())

gender: filled 41 nulls
occasions: filled 75 nulls
categories: filled 41 nulls
tags: filled 157 nulls
material: filled 15 nulls
short_description: filled 2 nulls

Total nulls remaining: 0


## 5. Drop Useless MRP Column
Every single row has `mrp = 0` — the column carries no information.

In [19]:
print('MRP unique values:', df['mrp'].unique())
df = df.drop(columns=['mrp'])
print('MRP column dropped. New shape:', df.shape)

MRP unique values: [0]
MRP column dropped. New shape: (2031, 20)


## 6. Standardize Gender & Strip Whitespace

In [20]:
# Capitalize gender consistently: Women / Men / Unisex / Unknown
df['gender'] = df['gender'].str.strip().str.capitalize()

# Strip whitespace from key columns
for col in ['name', 'sku', 'categories', 'occasions']:
    df[col] = df[col].str.strip()

print('Gender values after standardization:')
print(df['gender'].value_counts())

Gender values after standardization:
gender
Women      1615
Men         197
Unisex      178
Unknown      41
Name: count, dtype: int64


## 7. Final Validation & Save

In [21]:
print('=== FINAL SHAPE ===')
print(df.shape)

print('\n=== NULL CHECK ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls remaining')

print('\n=== GENDER DISTRIBUTION ===')
print(df['gender'].value_counts())

print('\n=== DUPLICATE SKUs ===')
print(df['sku'].duplicated().sum(), 'duplicates')

print('\n=== SAMPLE CLEANED ROW ===')
df[['sku', 'name', 'gender', 'occasions', 'categories']].head(3)

=== FINAL SHAPE ===
(2031, 20)

=== NULL CHECK ===
No nulls remaining

=== GENDER DISTRIBUTION ===
gender
Women      1615
Men         197
Unisex      178
Unknown      41
Name: count, dtype: int64

=== DUPLICATE SKUs ===
0 duplicates

=== SAMPLE CLEANED ROW ===


,sku,name,gender,occasions,categories
0,da23co005,Kora Mirri Co-ord Set,Women,"Office parties, Date night, Romantic Dinner, A...","Women, Sets, Co-ord set"
1,da23co004,Indigo Mirri Co-ord Set,Women,"Office parties, Date night, Romantic Dinner, A...","Women, Sets, Co-ord set"
2,da23to004,Red Mirri Shirt,Women,"Office parties, Date night, Romantic Dinner, A...","Women, Tops, Shirt"


In [22]:
df.to_csv('../docs/db_products_cleaned.csv', index=False)
print('Saved: db_products_cleaned.csv')
print('Rows:', len(df))
print('Columns:', len(df.columns))

Saved: db_products_cleaned.csv
Rows: 2031
Columns: 20


## 8. Fix Concatenated CamelCase Values in Occasions & Tags
Some rows have keywords jammed together like `BridesmaidWeddingSangeetMehendi` or `BlazerStyleShirtLenticular...`.
These are split on CamelCase boundaries into proper comma-separated values.

In [23]:
import re

def fix_concatenated(text):
    if pd.isna(text):
        return text
    text = str(text)
    tokens = re.split(r'[,;]+', text)
    fixed_tokens = []
    for token in tokens:
        token = token.strip()
        # Only split tokens that are long AND have CamelCase pattern
        if len(token) > 15 and re.search(r'[a-z][A-Z]', token):
            token = re.sub(r'([a-z])([A-Z])', r'\1, \2', token)
        fixed_tokens.append(token)
    return ', '.join(t.strip() for t in fixed_tokens if t.strip())

df['occasions'] = df['occasions'].apply(fix_concatenated)
df['tags'] = df['tags'].apply(fix_concatenated)

# Verify
def has_real_concat(text):
    words = str(text).split()
    return any(len(w) > 20 and '/' not in w and "'" not in w for w in words)

print('Remaining concat in occasions:', df['occasions'].apply(has_real_concat).sum())
print('Remaining concat in tags:', df['tags'].apply(has_real_concat).sum())
print('\nSample fixed occasion:')
print(df[df['sku']=='fi24w01']['occasions'].values[0][:200])

Remaining concat in occasions: 0
Remaining concat in tags: 8

Sample fixed occasion:
Bridesmaid, Wedding, Sangeet, Mehendi, Engagement, Haldi, Reception, Casual Outings, Brunch, Workwear, Travel, Resort Wear, Lounge Wear, Cocktail, Sundowner, Night Wear, Red Carpet, Dinner Date, Clubb
